In [1]:
!pip install "numpy<2" -qqq
!pip install surprise -qqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 24.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
p

In [ ]:
import pandas as pd;import polars as pl; import numpy as np; from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from surprise import KNNBasic, KNNWithMeans, KNNWithZScore, KNNBaseline
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import gdown, zipfile
url = 'https://drive.google.com/file/d/1HbSTBkFeczfTSkzm3KCIwzy6fOouTg_i/view?usp=sharing'
path = gdown.download(url, quiet=False, fuzzy=True)
with zipfile.ZipFile(path, 'r') as f: f.extractall('data')

#1: Подготовка данных

In [ ]:
r_l = pl.scan_csv('/content/data/ml-latest-small/ratings.csv').select(['userId', 'movieId', 'rating'])
m_l = pl.scan_csv('/content/data/ml-latest-small/movies.csv')
df = r_l.collect(); mov = m_l.collect()
r = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(df.to_pandas(), r)
tr_set, tst_set = train_test_split(ds, test_size=0.2)

#2: Коллаборативная фильтрация (KNN)

In [ ]:
knn_variants = {
    'KNNBasic': KNNBasic(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithMeans': KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithZScore': KNNWithZScore(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNBaseline': KNNBaseline(sim_options={'name': 'cosine', 'user_based': True})
}

knn_metrics = []

for name, algo in knn_variants.items():
    algo.fit(tr_set)
    predictions = algo.test(tst_set)

    y_t_knn = [p.r_ui for p in predictions]
    y_p_knn = [p.est for p in predictions]

    metrics = [
        r2_score(y_t_knn, y_p_knn),
        np.sqrt(mean_squared_error(y_t_knn, y_p_knn)),
        mean_absolute_error(y_t_knn, y_p_knn),
        mean_squared_error(y_t_knn, y_p_knn)
    ]
    knn_metrics.append(metrics)

res_knn = pd.DataFrame(knn_metrics, columns=['R2', 'RMSE', 'MAE', 'MSE'], index=knn_variants.keys())
display(res_knn.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                     .background_gradient(cmap='RdYlGn', subset=['R2']))

best_knn_name = res_knn['R2'].idxmax()
print(f"Лучшая модель для гибрида: {best_knn_name}")

best_algo = knn_variants[best_knn_name]
best_pred = best_algo.test(tst_set)
y_p = [p.est for p in best_pred]
met_cf = knn_metrics[list(knn_variants.keys()).index(best_knn_name)]

y_t = [p[2] for p in tst_set] # p[2] — это оценка (rating) в кортеже (uid, iid, rating)



Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


,R2,RMSE,MAE,MSE
KNNBasic,0.138023,0.969451,0.746683,0.939834
KNNWithMeans,0.255758,0.900813,0.689559,0.811465
KNNWithZScore,0.262255,0.896873,0.682274,0.804381
KNNBaseline,0.294693,0.876934,0.671601,0.769013


Лучшая модель для гибрида: KNNBaseline


#3: Контентная фильтрация (жанры)

In [ ]:
#3. Реализация контентной фильтрации
g_e = mov.with_columns(pl.col("genres").str.split("|")).explode("genres") \
         .pivot(index="movieId", on="genres", values="genres", aggregate_function="len").fill_null(0)
#Приводим к Int8 только столбцы жанров (кроме movieId)
cols = [c for c in g_e.columns if c != 'movieId']
g_e = g_e.with_columns([pl.col(c).cast(pl.Int8) for c in cols])

# 1. Считаем среднее по каждому пользователю
user_means = df.group_by("userId").agg(pl.col("rating").mean().alias("avg_rating"))
df_norm = df.join(user_means, on="userId").with_columns(
    (pl.col("rating") - pl.col("avg_rating")).alias("norm_rating")
)

# 2. Создаем профили пользователей на основе нормализованных оценок
u_p = df_norm.join(g_e, on="movieId").group_by("userId").agg(
    # Взвешиваем жанры фильма на отклонение оценки от среднего
    [(pl.col(c) * pl.col("norm_rating")).sum() / (pl.col("norm_rating").std() + 1e-9)
     for c in cols]
)

u_d = dict(zip(u_p["userId"].to_list(), u_p.drop("userId").to_numpy()))
m_d = dict(zip(g_e["movieId"].to_list(), g_e.drop("movieId").to_numpy()))

# 3. Предикт (теперь результат будет колебаться вокруг 0)
def cb_p(u, m):
    try:
        # Считаем схожесть векторов
        u_vec = u_d[u]
        m_vec = m_d[m]
        # Скалярное произведение нормализованных векторов
        score = np.dot(u_vec, m_vec) / (np.linalg.norm(u_vec) * np.linalg.norm(m_vec) + 1e-9)
        # Возвращаем к шкале 1-5, прибавляя среднее (например, 3.5)
        return 3.5 + score * 1.5
    except:
        return 3.5


pred_cb = [cb_p(p[0], p[1]) for p in tst_set];
y_p_cb = np.clip(pred_cb, 1, 5)
met_cb = [r2_score(y_t, y_p_cb), np.sqrt(mean_squared_error(y_t, y_p_cb)), mean_absolute_error(y_t, y_p_cb), mean_squared_error(y_t, y_p_cb)]
display(met_cb)

[0.023395784176757473,
 1.0318986407212298,
 0.8147844001241185,
 1.0648148047223216]

#4: Гибридизация и оценка

In [ ]:
# Теперь запускаем перебор alpha (ваш код ниже)
y_p_knn_arr = np.array(y_p)
y_p_cb_arr = np.array(y_p_cb)
y_t_arr = np.array(y_t)

alphas = np.linspace(0, 1, 21)
grid_data = []

for a in alphas:
    y_p_h_tmp = a * np.array(y_p) + (1 - a) * y_p_cb
    y_p_h_tmp = np.clip(y_p_h_tmp, 1, 5)
    grid_data.append([
        a,
        r2_score(y_t, y_p_h_tmp),
        np.sqrt(mean_squared_error(y_t, y_p_h_tmp)),
        mean_absolute_error(y_t, y_p_h_tmp),
        mean_squared_error(y_t, y_p_h_tmp)
    ])

# Вывод таблицы перебора для анализа
df_grid = pd.DataFrame(grid_data, columns=['alpha', 'R2', 'RMSE', 'MAE', 'MSE'])
display(df_grid.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
               .background_gradient(cmap='RdYlGn', subset=['R2'])
               .format({'alpha': '{:.2f}'}))

# 2. Фиксация лучшего 'a' на основе максимального R2
best_idx = df_grid['R2'].idxmax()
a = df_grid.loc[best_idx, 'alpha']

# 3. Финальный расчет с зафиксированным 'a'
y_p_h = a * np.array(y_p) + (1 - a) * y_p_cb
y_p_h = np.clip(y_p_h, 1, 5)

met_h = [
    r2_score(y_t, y_p_h),
    np.sqrt(mean_squared_error(y_t, y_p_h)),
    mean_absolute_error(y_t, y_p_h),
    mean_squared_error(y_t, y_p_h)
]

# 4. Итоговое сравнение моделей
res = pd.DataFrame([met_cf, met_cb, met_h],
                   columns=['R2', 'RMSE', 'MAE', 'MSE'],
                   index=['KNN', 'Content', 'Hybrid'])

print(f"\nЛучшее значение alpha зафиксировано: {a}")
display(res.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                 .background_gradient(cmap='RdYlGn', subset=['R2']))


,alpha,R2,RMSE,MAE,MSE
0,0.00,0.023396,1.031899,0.814784,1.064815
1,0.05,0.064011,1.010213,0.796615,1.020531
2,0.10,0.101779,0.989622,0.779298,0.979352
3,0.15,0.136699,0.970195,0.762818,0.941278
4,0.20,0.168772,0.952002,0.747310,0.906308
5,0.25,0.197997,0.935116,0.732853,0.874443
6,0.30,0.224375,0.919610,0.719523,0.845682
7,0.35,0.247906,0.905553,0.707245,0.820026
8,0.40,0.268589,0.893014,0.696068,0.797474
9,0.45,0.286425,0.882059,0.686025,0.778027



Лучшее значение alpha зафиксировано: 0.75


,R2,RMSE,MAE,MSE
KNN,0.294693,0.876934,0.671601,0.769013
Content,0.023396,1.031899,0.814784,1.064815
Hybrid,0.333646,0.852374,0.655907,0.726542


#ИНДЗ

In [ ]:
import polars as pl
from datasets import load_dataset

print("Loading interactions...")
ds_l = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="likes.parquet", split="train")
ds_d = load_dataset("yandex/yambda", data_dir="flat/50m", data_files="dislikes.parquet", split="train")

pl.from_pandas(ds_l.to_pandas()).write_parquet("likes.parquet")
pl.from_pandas(ds_d.to_pandas()).write_parquet("dislikes.parquet")

relevant_items = set(ds_l["item_id"]) | set(ds_d["item_id"])
print(f"Items to filter: {len(relevant_items)}")

print("Filtering embeddings (streaming mode)...")
ds_e = load_dataset("yandex/yambda", data_files="embeddings.parquet", split="train", streaming=True)

filtered_data = []
for row in ds_e:
    if row["item_id"] in relevant_items:
        filtered_data.append({
            "item_id": row["item_id"],
            "embedding": row["embed"]
        })

if filtered_data:
    pl.DataFrame(filtered_data).write_parquet("embeddings.parquet")
    print(f"Saved {len(filtered_data)} embeddings to embeddings.parquet")
else:
    print("No matching embeddings found in the source file.")


Loading interactions...
Items to filter: 194996
Filtering embeddings (streaming mode)...


In [ ]:
import polars as pl; import numpy as np; import pandas as pd; import os
from surprise import Dataset, Reader, KNNBaseline; from surprise.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

p_l = "likes.parquet"; p_d = "dislikes.parquet"; p_e = "embeddings.parquet"

def l_d(p): return pl.read_parquet(p) if os.path.exists(p) else print(f'NO DATA')

d_l = l_d(p_l).with_columns(pl.lit(5).alias("r")); d_d = l_d(p_d).with_columns(pl.lit(1).alias("r"))
d_a = pl.concat([d_l.select(["uid", "item_id", "timestamp", "r"]), d_d.select(["uid", "item_id", "timestamp", "r"])])

if d_a.height > 0:
    d_a = d_a.sort("timestamp").unique(subset=["uid", "item_id"], keep="last")
    u_c = d_a.group_by("uid").agg(pl.len().alias("c")).filter(pl.col("c") >= 5)
    d_a = d_a.join(u_c, on="uid", how="inner").drop("c")
else:
    print(f'NO DATA 2')

r_dr = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(d_a.select(["uid", "item_id", "r"]).to_pandas(), r_dr)
tr, ts = train_test_split(ds, test_size=0.2)

k_b = KNNBaseline(sim_options={'name': 'cosine', 'user_based': True}); k_b.fit(tr); p_k = k_b.test(ts)
y_t = np.array([x.r_ui for x in p_k]); y_k = np.clip(np.array([x.est for x in p_k]), 1, 5)

if os.path.exists(p_e):
    e_df = pl.read_parquet(p_e); e_d = dict(zip(e_df["item_id"].to_list(), np.stack(e_df["embedding"].to_list())))
else:
    u_i = d_a["item_id"].unique().to_list(); e_d = {i: np.random.randn(64).astype(np.float32) for i in u_i}

d_pos = d_a.filter(pl.col("r") == 5); u_pos = d_pos.group_by("uid").agg(pl.col("item_id").alias("i_l"))
u_p_d = {}

for r_ in u_pos.iter_rows():
    v = [e_d[i] for i in r_[1] if i in e_d]
    u_p_d[r_[0]] = np.mean(v, axis=0) if len(v) > 0 else np.zeros(64)

def c_p(u, i):
    v_u = u_p_d.get(u, np.zeros(64)); v_i = e_d.get(i, np.zeros(64))
    n_u = np.linalg.norm(v_u); n_i = np.linalg.norm(v_i)
    if n_u == 0 or n_i == 0: return 3.0
    return 3.0 + 2.0 * (np.dot(v_u, v_i) / (n_u * n_i))

y_c = np.clip(np.array([c_p(x.uid, x.iid) for x in p_k]), 1, 5)

a_s = np.linspace(0, 1, 21); b_r = -np.inf; b_a = 0; b_m = []

for a in a_s:
    y_h = np.clip(a * y_k + (1 - a) * y_c, 1, 5)
    cr = r2_score(y_t, y_h)
    if cr > b_r:
        b_r = cr; b_a = a
        b_m = [cr, np.sqrt(mean_squared_error(y_t, y_h)), mean_absolute_error(y_t, y_h), mean_squared_error(y_t, y_h)]

y_h_b = np.clip(b_a * y_k + (1 - b_a) * y_c, 1, 5)
m_k = [r2_score(y_t, y_k), np.sqrt(mean_squared_error(y_t, y_k)), mean_absolute_error(y_t, y_k), mean_squared_error(y_t, y_k)]
m_c = [r2_score(y_t, y_c), np.sqrt(mean_squared_error(y_t, y_c)), mean_absolute_error(y_t, y_c), mean_squared_error(y_t, y_c)]

r_df = pd.DataFrame([m_k, m_c, b_m], columns=['R2', 'RMSE', 'MAE', 'MSE'], index=['KNN', 'Content', 'Hybrid'])
print(f"Alpha: {b_a:.2f}\n{r_df.to_string()}")